# RABC in Weaviate

This recipe will show how to implement a simple RABC for Weaviate.

We can create roles to manage different resource types:
- Roles
- Collections
- Tenants
- Objects
- Backup

Our objective with this recipe is:

Create two new roles: `Developer` and `Intern`.
Create two users: `Developer1` and `Intern1`, that will belong to their respective roles.

Our Developer user will only be able to `create`, `read`, `update` and `delete` resources starting with `dev*`, while our `intern` will only be able to read those resources

For more information on rbac, visit: https://weaviate.io/developers/weaviate/configuration/rbac

In [ ]:
!pip install -Uq weaviate-client

In [1]:

import weaviate
client = weaviate.connect_to_local(
    auth_credentials=weaviate.classes.init.Auth.api_key("root-user-key")
)
# lets print the client and server version
print(f"Client: {weaviate.__version__}, Server: {client.get_meta().get('version')}")

Client: 4.20.4, Server: 1.36.8


In [2]:
# As Root we should be able to already list all collections, but should have none for now
client.collections.list_all()

{}

## Let's check our default roles, their permissions and actions

In [12]:
for role_name, role in client.roles.list_all().items():
    print("#####")
    print("role_name:", role_name)
    print("Alias Permissions:", role.alias_permissions)
    print("Collection Permissions:", role.collections_permissions)
    print("Backup Permissions:", role.backups_permissions)
    print("Cluster Permissions:", role.cluster_permissions)
    print("Data Permissions:", role.data_permissions)
    print("Groups Permissions:", role.groups_permissions)
    print("Nodes Permissions:", role.nodes_permissions)
    print("Replicate Permissions:", role.replicate_permissions)
    print("Roles Permissions:", role.roles_permissions)
    print("Tenants Permissions:", role.tenants_permissions)
    print("\n")

#####
role_name: admin
Alias Permissions: [AliasPermissionOutput(actions={<AliasAction.DELETE: 'delete_aliases'>, <AliasAction.UPDATE: 'update_aliases'>, <AliasAction.READ: 'read_aliases'>, <AliasAction.CREATE: 'create_aliases'>}, alias='*', collection='*')]
Collection Permissions: [CollectionsPermissionOutput(actions={<CollectionsAction.DELETE: 'delete_collections'>, <CollectionsAction.READ: 'read_collections'>, <CollectionsAction.CREATE: 'create_collections'>, <CollectionsAction.UPDATE: 'update_collections'>}, collection='*')]
Backup Permissions: [BackupsPermissionOutput(actions={<BackupsAction.MANAGE: 'manage_backups'>}, collection='*')]
Cluster Permissions: [ClusterPermissionOutput(actions={<ClusterAction.READ: 'read_cluster'>})]
Data Permissions: [DataPermissionOutput(actions={<DataAction.DELETE: 'delete_data'>, <DataAction.UPDATE: 'update_data'>, <DataAction.READ: 'read_data'>, <DataAction.CREATE: 'create_data'>}, collection='*', tenant='*')]
Groups Permissions: [GroupsPermission

## Let's list all root on this cluster

In [13]:
assigned_users = client.roles.get_user_assignments(role_name="root")

for user in assigned_users:
    print(user)

UserAssignment(user_id='root-user', user_type=<UserTypes.DB_STATIC: 'db_env_user'>)


In [14]:
# Let's list our users
print(client.users.db.list_all())

[UserDB(user_id='root-user', role_names=['root'], user_type=<UserTypes.DB_STATIC: 'db_env_user'>, active=True, created_at=datetime.datetime(1, 1, 1, 0, 0, tzinfo=datetime.timezone.utc), last_used_time=None, api_key_first_letters=None)]


In [17]:
# creating the developer user

developer_api_key = client.users.db.create(user_id="developer1")
print(developer_api_key)

UlZlMHlDdXRuMm9ZK2R2el9WV2lQcmxjSndRbGZNSTMwTVUxWjM4YnBxV0RuL0FaSnZkeWwrYWQ2VW9NPV92MjAw


In [15]:

# creating the intern user
intern_api_key = client.users.db.create(user_id="intern1")
print(intern_api_key)

SkcvRFpvUW1sbWdONVprUF9qalVIc2ZBNFJscHpRcDJuZXNROERvamxRUmpuNW5xYTBXUVRFNXZVWjZVPV92MjAw


In [18]:
# let's list our users
for user in client.users.db.list_all():
    print("#####")
    print(user)

#####
UserDB(user_id='developer1', role_names=[], user_type=<UserTypes.DB_DYNAMIC: 'db_user'>, active=True, created_at=datetime.datetime(2026, 4, 6, 15, 0, 2, 688000, tzinfo=datetime.timezone.utc), last_used_time=None, api_key_first_letters='UlZ')
#####
UserDB(user_id='intern1', role_names=[], user_type=<UserTypes.DB_DYNAMIC: 'db_user'>, active=True, created_at=datetime.datetime(2026, 4, 6, 14, 59, 42, 967000, tzinfo=datetime.timezone.utc), last_used_time=None, api_key_first_letters='Skc')
#####
UserDB(user_id='root-user', role_names=['root'], user_type=<UserTypes.DB_STATIC: 'db_env_user'>, active=True, created_at=datetime.datetime(1, 1, 1, 0, 0, tzinfo=datetime.timezone.utc), last_used_time=None, api_key_first_letters=None)


In [41]:
# Now, let's create our developer role
from weaviate.classes.rbac import Permissions

permissions = [
    # to our collections resources, we all permission only for *dev collections
    Permissions.collections(
        collection="dev*",  # Applies to all collections starting with "dev"
        create_collection=True,  # Allow creating new collections
        read_config=True,  # Allow reading collection info/metadata
        update_config=True,  # Allow updating collection configuration, i.e. update schema properties, when inserting data with new properties
        delete_collection=True,  # Allow deleting collections
    ),
    # We also want to limit the acess to tenants
    Permissions.tenants(
        collection="dev*",  # Applies to all collections starting with "dev"
        tenant="dev*",  # Applies to all tenants starting with "dev"
        create=True,  # Allow creating new tenants
        read=True,  # Allow reading tenant info/metadata
        update=True,  # Allow updating tenant states
        delete=True,  # Allow deleting tenants
    ),
    # at the data graunalrity, those are the permissions we want to give
    Permissions.data(
        collection="dev*",  # Applies to all collections starting with "dev"
        tenant="*",  # Applies to all tenants starting with "dev"
        create=True,  # Allow data inserts
        read=True,  # Allow query and fetch operations
        update=True,  # Allow data updates
        delete=False,  # Allow data deletes
    ),
    #
    Permissions.backup(
        collection="dev*",  # Applies to all collections starting with "dev"
        manage=True,  # Allow managing backups
    ),    
]

client.roles.create(role_name="developer", permissions=permissions)

Role(name='developer', alias_permissions=[], cluster_permissions=[], collections_permissions=[CollectionsPermissionOutput(actions={<CollectionsAction.READ: 'read_collections'>, <CollectionsAction.CREATE: 'create_collections'>, <CollectionsAction.DELETE: 'delete_collections'>, <CollectionsAction.UPDATE: 'update_collections'>}, collection='Dev*')], data_permissions=[DataPermissionOutput(actions={<DataAction.UPDATE: 'update_data'>, <DataAction.READ: 'read_data'>, <DataAction.CREATE: 'create_data'>}, collection='Dev*', tenant='*')], roles_permissions=[], users_permissions=[], backups_permissions=[BackupsPermissionOutput(actions={<BackupsAction.MANAGE: 'manage_backups'>}, collection='Dev*')], nodes_permissions=[], tenants_permissions=[TenantsPermissionOutput(actions={<TenantsAction.DELETE: 'delete_tenants'>, <TenantsAction.UPDATE: 'update_tenants'>, <TenantsAction.CREATE: 'create_tenants'>, <TenantsAction.READ: 'read_tenants'>}, collection='Dev*', tenant='dev*')], replicate_permissions=[], 

In [20]:
# great, let's list our roles
for role in client.roles.list_all():
    print("#####")
    print(role)

#####
admin
#####
developer
#####
read-only
#####
root
#####
viewer


In [42]:
# let's inspect our role
test_role = client.roles.get(role_name="developer")

print(test_role)
print("-")
print(test_role.collections_permissions)
print("-")
print(test_role.data_permissions)

Role(name='developer', alias_permissions=[], cluster_permissions=[], collections_permissions=[CollectionsPermissionOutput(actions={<CollectionsAction.READ: 'read_collections'>, <CollectionsAction.CREATE: 'create_collections'>, <CollectionsAction.DELETE: 'delete_collections'>, <CollectionsAction.UPDATE: 'update_collections'>}, collection='Dev*')], data_permissions=[DataPermissionOutput(actions={<DataAction.UPDATE: 'update_data'>, <DataAction.READ: 'read_data'>, <DataAction.CREATE: 'create_data'>}, collection='Dev*', tenant='*')], roles_permissions=[], users_permissions=[], backups_permissions=[BackupsPermissionOutput(actions={<BackupsAction.MANAGE: 'manage_backups'>}, collection='Dev*')], nodes_permissions=[], tenants_permissions=[TenantsPermissionOutput(actions={<TenantsAction.DELETE: 'delete_tenants'>, <TenantsAction.UPDATE: 'update_tenants'>, <TenantsAction.CREATE: 'create_tenants'>, <TenantsAction.READ: 'read_tenants'>}, collection='Dev*', tenant='dev*')], replicate_permissions=[], 

In [22]:
# now, we do the same for our intern, but limited to read only
intern_roles = [
    # to our collections resources, we give all permission only for *dev collections
    Permissions.collections(
        collection="dev*", # for all collections dev*
        create_collection=False, # cant create a new one
        read_config=True,  # can read the config
        update_config=False,  # no update
        delete_collection=False,  # definitely no deletion!
    ),
    # We also want to limit the acess to tenants
    Permissions.tenants(
        collection="dev*",  # Applies to all collections starting with "dev"
        tenant="dev*",  # Applies to all tenants in collections starting with "dev"
        create=False,  # no new tenants
        read=True,  # ok to read
        update=False,  # no update
        delete=False,  # no delete
    ),
    # at the data granularity, those are the permissions we want to give
    Permissions.data(
        collection="dev*",  # Applies to all collections starting with "dev"
        tenant="dev*",  # Applies to all tenants starting with "dev"
        create=False,  # Allow data inserts
        read=True,  # Allow query and fetch operations
        update=False,  # Allow data updates
        delete=False,  # Allow data deletes
    ),
]
client.roles.create(role_name="intern", permissions=intern_roles)

Role(name='intern', alias_permissions=[], cluster_permissions=[], collections_permissions=[CollectionsPermissionOutput(actions={<CollectionsAction.READ: 'read_collections'>}, collection='Dev*')], data_permissions=[DataPermissionOutput(actions={<DataAction.READ: 'read_data'>}, collection='Dev*', tenant='dev*')], roles_permissions=[], users_permissions=[], backups_permissions=[], nodes_permissions=[], tenants_permissions=[TenantsPermissionOutput(actions={<TenantsAction.READ: 'read_tenants'>}, collection='Dev*', tenant='dev*')], replicate_permissions=[], groups_permissions=[])

In [23]:
# let's inspect our role
test_role = client.roles.get(role_name="intern")

print(test_role)
print("-")
print(test_role.collections_permissions)
print("-")
print(test_role.data_permissions)

Role(name='intern', alias_permissions=[], cluster_permissions=[], collections_permissions=[CollectionsPermissionOutput(actions={<CollectionsAction.READ: 'read_collections'>}, collection='Dev*')], data_permissions=[DataPermissionOutput(actions={<DataAction.READ: 'read_data'>}, collection='Dev*', tenant='dev*')], roles_permissions=[], users_permissions=[], backups_permissions=[], nodes_permissions=[], tenants_permissions=[TenantsPermissionOutput(actions={<TenantsAction.READ: 'read_tenants'>}, collection='Dev*', tenant='dev*')], replicate_permissions=[], groups_permissions=[])
-
[CollectionsPermissionOutput(actions={<CollectionsAction.READ: 'read_collections'>}, collection='Dev*')]
-
[DataPermissionOutput(actions={<DataAction.READ: 'read_data'>}, collection='Dev*', tenant='dev*')]


In [43]:
# now it's time to assign our roles to the users
client.users.db.assign_roles(user_id="developer1", role_names=["developer"])
client.users.db.assign_roles(user_id="intern1", role_names=["intern"])

In [27]:
# great! Previously we have checked all users in a role.
# let's to the inverse: check all roles for a user
user_intern_roles = client.users.db.get_assigned_roles(user_id="intern1")
print(user_intern_roles)

{'intern': RoleBase(name='intern')}


In [35]:
# now let's create a production collection. None of our users should
# be able to access it.
col = client.collections.create("ProductionCollection")
col.data.insert({"text": "hello this is Production!"}, vector=[1, 2, 3])

UUID('9367dff7-0d28-451b-85c0-6a9e7de9eb5b')

In [36]:
# Now, let's create a dev collection. Only the developer should be able to access it.
dev_collection = client.collections.create("devCollection")
dev_collection.data.insert({"text": "hello this is dev!"}, vector=[1, 2, 3])

UUID('3ff7c5c3-e00f-4794-ab65-a5a89a3c31ec')

In [37]:
# now let's instantiate two new clients, using users credentials
developer_client = weaviate.connect_to_local(
    auth_credentials=weaviate.classes.init.Auth.api_key(developer_api_key)
)
intern_client = weaviate.connect_to_local(
    auth_credentials=weaviate.classes.init.Auth.api_key(intern_api_key)
)

In [44]:
developer_client.collections.use("devCollection").data.insert({"text": "hello this is dev from developer!"}, vector=[1, 2, 3])

UUID('465ebe60-1ae3-48d5-89e8-67d1ba9a641f')

In [51]:
try:
    intern_client.collections.use("devCollection").data.insert({"text": "hello this is dev from intern!"}, vector=[1, 2, 3])
except weaviate.exceptions.InsufficientPermissionsError as e:
    print("Intern does not have permissions to insert data into devCollection:", e)

Intern does not have permissions to insert data into devCollection: forbidden! Unexpected status code: 403, with response body: {'error': [{'message': "rbac: authorization, forbidden action: user 'intern1' has insufficient permissions to create_data [[Domain: data, Collection: DevCollection, Tenant: *, Object: *]]"}]}.


In [55]:
# both intern and developer will only have access to dev* collections
intern_client.collections.list_all().keys() == developer_client.collections.list_all().keys()

True